# Access PACE data with OPeNDAP

In [ ]:
import xarray as xr
import datetime as dt
import earthaccess
import matplotlib.pyplot as plt
import numpy as np
# import pydap-specific tools
from pydap.client import get_cmr_urls, open_url
from pydap.client import to_netcdf as dap_to_netcdf

In [ ]:
auth = earthaccess.login(strategy="netrc", persist=True) # you will be promted to add your EDL credentials

# pass Token Authorization to a new Session.
my_session = session=auth.get_session()

In [ ]:
PACE_ccid = "C3620140256-OB_CLOUD" # version 3.1 
time_range=[dt.datetime(2025, 1, 1), dt.datetime(2025, 3, 30)]
cmr_urls = get_cmr_urls(ccid=PACE_ccid, limit=1000, time_range=time_range) # limit by default = 50

print("################################################ \n We found a total of ", len(cmr_urls), "OPeNDAP URLS!!!\n################################################")

In [ ]:
cmr_urls[:5]

## Identify only those granules associated with 4km resolution


In [ ]:
chlor_a_urls = [url for url in cmr_urls if "DAY.CHL.V3_1.chlor_a.4km" in url]
len(chlor_a_urls)

In [ ]:
chlor_a_urls[0]

In [ ]:
ds = open_url(chlor_a_urls[0], session=my_session, protocol="dap4")
ds.tree()

## Spatial subset
```
# Min/max of lon values
minLon, maxLon = -96, 10

# Min/Max of lat values
minLat, maxLat = 6, 70
```

In [ ]:
ds = xr.open_dataset(chlor_a_urls[0].replace("https","dap4"), session=my_session, engine='pydap')
ds

In [ ]:
# Min/max of lon values
minLon, maxLon = -96, 10
# Min/Max of lat values
minLat, maxLat = 6, 70

lat, lon = ds['lat'].values, ds['lon'].values

iLon = np.where((lon>minLon)&(lon < maxLon))[0]
iLat= np.where((lat>minLat)&(lat < maxLat))[0]


In [ ]:
output_path = "data/"

keep_variables = ['/lon', '/lat', "/chlor_a"]
dim_slices = {'/lat': (iLat[0], iLat[-1]), '/lon': (iLon[0], iLon[-1])}


## Stream data

In [ ]:
%%time
dap_to_netcdf(chlor_a_urls, session=my_session, keep_variables=keep_variables, dim_slices=dim_slices, output_path=output_path)

## Inspect local files

In [ ]:
nds = xr.open_mfdataset(output_path+"PACE_OCI.*.nc4", concat_dim='time', parallel=True, combine="nested")
nds

In [ ]:
vmin = ds['chlor_a'].attrs['display_min']
vmax = ds['chlor_a'].attrs['display_max']

In [ ]:
chlor_a.plot.contourf(vmin=vmin, vmax=vmax, levels=100, cmap='RdBu_r');